In [43]:
import requests
import pandas as pd
import lxml
from io import StringIO

In [44]:
BASE_URL = "https://retro.umoiq.com/service/publicXMLFeed?a=unitrans"
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36"}

In [45]:
def getRoutes() -> pd.Series:
    """
    Get all available routes as a pandas df.
    Routes aren't expected to change any time soon, so this won't be saved.
    """
    response = requests.get(BASE_URL, headers=HEADERS, params = {"command" : "routeList"})
    return pd.read_xml(StringIO(response.text))["tag"]

In [46]:
def getSchedules(routeList: pd.Series) -> dict:
    """
    Iterate over a series of routes, and get a hashmap with the key being the route and the schedule dataframe being the value.
    We'll parse this later, since pandas will struggle to clean this up...
    This also doesn't change by time, so we don't need to save this.
    """
    scheduleData = {}
    for row in routeList:
        scheduleData[row] = requests.get(BASE_URL, headers=HEADERS, params = {"command":"schedule", "r":row}).text
    return scheduleData


In [47]:
# Get stopids for "Silo" and "Memorial Union"
import re

SEARCH_TERMS = ["Silo", "Memorial Union"]
stops = []
schedules = getSchedules(getRoutes())
for route, xml in schedules.items():
    for line in xml.splitlines():
        for term in SEARCH_TERMS:
            if term in line:
                # Use regex to pull the line number out
                match = re.search("tag=\"22([0-9]*)", line)
                if match:
                    id = match.group(1)
                    stops.append(id)

stops = pd.Series(stops).unique()

# Manual correction: Stop 167 doesn't exist. Remove
stops = stops[stops != "167"]

In [ ]:
file_base = "../data/schedules/unitrans_route_"

# saves all the files
for k,v in schedules.items():
    with open(f"{file_base}{k}.xml", "w") as f:
        f.write(v)

In [49]:
from datetime import datetime
import os
import concurrent.futures, threading
TERMINUS = list(stops)
thread_local = threading.local()

def getSession():
    """
    Create new requests session for each thread
    """
    if not hasattr(thread_local, "session"):
        thread_local.session = requests.Session()
    return thread_local.session

def getAndWriteStop(stop):
    """
    Get predictions for stop 272,273,274,256 and 258. These are the main terminus stops.
    Retrieval is non-blocking and multithreaded.
    """
    session = getSession()
    
    url = f'{BASE_URL}'
    stopData = session.get(BASE_URL, headers = HEADERS, params = {"command":"predictions", "stopId":stop}).text
    # Write predictions to a timestamped file.
    # Calculate time
    now = datetime.now()
    folderpath = f'../data/predictions/{now.strftime("%m_%d_%y")}'
    mmddyyyy = now.strftime("%m_%d_%y_%H:%M")
    if not os.path.exists(folderpath):
        os.mkdir(folderpath)

    fname = f'{folderpath}/{mmddyyyy}-stop-{stop}.xml'

    with open(fname, 'a') as file:
        print(f'Wrote {fname}')
        file.write(stopData)

def getPredictions():
    """
    Call the helper functions and write the predictions to disk.
    """
    with concurrent.futures.ThreadPoolExecutor(max_workers = os.cpu_count()) as executor:
        executor.map(getAndWriteStop,TERMINUS)
    
    

In [50]:
if __name__ == '__main__':
    getPredictions()

Wrote ../data/predictions/03_07_26/03_07_26_13:22-stop-272.xml
Wrote ../data/predictions/03_07_26/03_07_26_13:22-stop-258.xml
Wrote ../data/predictions/03_07_26/03_07_26_13:22-stop-274.xml
Wrote ../data/predictions/03_07_26/03_07_26_13:22-stop-273.xml
Wrote ../data/predictions/03_07_26/03_07_26_13:22-stop-256.xml


This file is converted into a .py with jupyter nbconvert --to python and run every 5 minutes from 6AM to 11PM with a crontab.

According to unitrans.ucdavis.edu/general-information, service starts from 6:30 and ends at 10:40. However, there appear to be a few lines that disobey this rule, so we'll start recording at 6am and end at 11pm (1020 minutes).

Recordings made every 5 minutes, since the arrival times are always on a 5, but not otherwise consistent.

This means 204 recordings a day * 6 stops * 7 day = 8586 files in total for a week.

Caching is undesired, since we are querying the same bus/lines, but want new observations every time
Plus, we are querying per hour, not real-time.

Based on API documentation from https://retro.umoiq.com/xmlFeedDocs/NextBusXMLFeed.pdf
Documentation discovered from scraping UC Davis Unitrans site's network requests